In [ ]:
# Cell 1: Setup & Environment
import os
import sys
import time
import platform
from pathlib import Path
import numpy as np
import torch

# Detect environment and configure sys.path
is_kaggle = Path("/kaggle/working").exists()
work_dir = Path("/kaggle/working") if is_kaggle else Path("output/kaggle_phase2c").resolve()
work_dir.mkdir(parents=True, exist_ok=True)

shards_dir = work_dir / "shards"
reports_dir = work_dir / "reports"
shards_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Locate repo or Kaggle dataset packages
code_candidates = [
    Path("/kaggle/input/wf052-code/src"),
    Path("/kaggle/input/wf052-code"),
    Path("src").resolve(),
    Path("../src").resolve(),
]
for c in code_candidates:
    if c.exists() and str(c) not in sys.path:
        sys.path.insert(0, str(c))

tiles_candidates = [
    Path("/kaggle/input/wf052-tiles-train"),
    Path("data/tiles/fires").resolve(),
    Path("../data/tiles/fires").resolve(),
]
fires_dir = None
for t in tiles_candidates:
    if (t / "data" / "tiles" / "fires").exists():
        fires_dir = t / "data" / "tiles" / "fires"
        break
    elif t.exists() and (t / "caldor").exists():
        fires_dir = t
        break

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
if device.type == "cuda":
    hw_name = f"NVIDIA {torch.cuda.get_device_name(0)}"
elif device.type == "mps":
    hw_name = "Apple Silicon (MPS)"
else:
    hw_name = f"CPU ({platform.processor() or platform.machine()})"

print("=" * 60)
print("Cell 1: Environment & Hardware Configuration")
print("=" * 60)
print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version:    {torch.version.cuda}")
    print(f"Device Count:    {torch.cuda.device_count()}")
print(f"Active Device:   {device} ({hw_name})")
print(f"Working Dir:     {work_dir}")
print(f"Fires Tiles Dir: {fires_dir}")
print("=" * 60)


In [ ]:
# Cell 2: Teacher Run Timing on Full 256x256 Tile (12 h Horizon)
import json
from wf052.surrogate.standin_generator import load_fire_rasters_crop, simulate_cellular_standin_arrival, fbfm40_to_fuel_group
from wf052.surrogate.dataset import calculate_upwind_third_ignition
from wf052.surrogate.latency import _sync_device

print("=" * 60)
print("Cell 2: Benchmarking Physics Teacher Latency Baseline")
print("=" * 60)

# Load 256x256 test crop from Caldor fire
fire_ref = "caldor"
fbfm_base, elev_base, canopy_base, wind_base = load_fire_rasters_crop(
    fire_name=fire_ref,
    tile_size=(256, 256),
    fires_base_dir=str(fires_dir) if fires_dir else "data/tiles/fires",
)
fuel_grp = fbfm40_to_fuel_group(fbfm_base)
ign_r, ign_c = calculate_upwind_third_ignition(wind_base, tile_shape=(256, 256))

# Warm-up teacher runs (3 runs)
for _ in range(3):
    _ = simulate_cellular_standin_arrival(
        fuel_group=fuel_grp,
        hourly_wind_uv=wind_base,
        ign_r=ign_r,
        ign_c=ign_c,
        horizon_min=720.0,
    )

# Timed teacher runs (10 runs)
teacher_timings_ms = []
for _ in range(10):
    t0 = time.perf_counter()
    arr, cens = simulate_cellular_standin_arrival(
        fuel_group=fuel_grp,
        hourly_wind_uv=wind_base,
        ign_r=ign_r,
        ign_c=ign_c,
        horizon_min=720.0,
    )
    teacher_timings_ms.append((time.perf_counter() - t0) * 1000.0)

t_arr = np.array(teacher_timings_ms)
teacher_mean_ms = float(np.mean(t_arr))
teacher_median_ms = float(np.median(t_arr))
teacher_p95_ms = float(np.percentile(t_arr, 95))

print(f"Teacher Simulation Timing (256x256, 12 h horizon):")
print(f"  Mean:   {teacher_mean_ms:.2f} ms")
print(f"  Median: {teacher_median_ms:.2f} ms")
print(f"  P95:    {teacher_p95_ms:.2f} ms")

with open(reports_dir / "teacher_latency_baseline.json", "w", encoding="utf-8") as f:
    json.dump({
        "hardware": hw_name,
        "fire_reference": fire_ref,
        "tile_size": [256, 256],
        "horizon_hours": 12.0,
        "mean_ms": teacher_mean_ms,
        "median_ms": teacher_median_ms,
        "p95_ms": teacher_p95_ms,
    }, f, indent=2)
print("Saved teacher baseline to reports/teacher_latency_baseline.json")


In [ ]:
# Cell 3: Resumable Shard Generation (100 samples per shard to /kaggle/working/shards/)
import pickle
from wf052.surrogate.standin_generator import generate_ensemble_samples

print("=" * 60)
print("Cell 3: Resumable Shard Generation")
print("=" * 60)

TOTAL_SHARDS = 5          # 5 shards = 500 samples (adjust as needed on Kaggle GPU)
SAMPLES_PER_SHARD = 100
fires_list = ["bobcat", "caldor", "creek", "glass", "kincade", "tamarack", "zogg"]

for shard_idx in range(TOTAL_SHARDS):
    shard_file = shards_dir / f"shard_{shard_idx:03d}.pkl"
    if shard_file.exists():
        print(f"Shard {shard_idx:03d} already exists at {shard_file.name} ({shard_file.stat().st_size:,} bytes). Skipping (resumed).")
        continue

    print(f"Generating shard {shard_idx:03d} ({SAMPLES_PER_SHARD} samples) ...", end=" ", flush=True)
    t_start = time.perf_counter()

    # Generate 100 teacher_v1 samples across 7 training fires with sampled HRRR wind series
    members_per_fire = int(np.ceil(SAMPLES_PER_SHARD / len(fires_list)))
    shard_samples = generate_ensemble_samples(
        fire_names=fires_list,
        num_members=members_per_fire,
        base_seed=42000 + shard_idx * 1000,
        tile_size=(256, 256),
        source_mode="teacher_v1_cellular",  # Teacher labels only
    )[:SAMPLES_PER_SHARD]

    # Save atomically to shard
    tmp_shard = shards_dir / f"tmp_shard_{shard_idx:03d}.pkl"
    with open(tmp_shard, "wb") as f:
        pickle.dump(shard_samples, f, protocol=pickle.HIGHEST_PROTOCOL)
    tmp_shard.rename(shard_file)

    elapsed = time.perf_counter() - t_start
    print(f"Done in {elapsed:.1f}s ({len(shard_samples)} samples saved to {shard_file.name}).")

print("All shards verified present in /kaggle/working/shards/")


In [ ]:
# Cell 4: Run Diagnostics (Steps 1-4) and Model Checks (Steps 5-7)
import pickle
from wf052.surrogate.dataset import SurrogateSample
from wf052.surrogate.trainer import SurrogateTrainer, TrainConfig
from wf052.surrogate.standin_generator import (
    compute_stratified_histogram,
    stratify_samples_by_burned_fraction,
    format_stratification_report,
)
from wf052.surrogate.adapter import compute_burn_head_miss_rates
from wf052.surrogate.calibration import OneSidedLowerConformalCalibrator
from wf052.surrogate.latency import benchmark_surrogate_latency, run_whole_domain_inference, verify_size_dependence
from wf052.surrogate.model import WindConditionedUNet

print("=" * 60)
print("Cell 4: Phase 2C Diagnostics & Verification Checks")
print("=" * 60)

# Load samples from all generated shards
all_samples = []
for s_path in sorted(shards_dir.glob("shard_*.pkl")):
    with open(s_path, "rb") as f:
        all_samples.extend(pickle.load(f))
print(f"Loaded {len(all_samples)} total samples from shards.")

# Check 1: Verify teacher_v1 labels only; trainer rejects ellipse baseline samples
print("\n--- Check 1: Training Labels Constraint ---")
dummy_model = WindConditionedUNet(base_channels=16)
try:
    # Synthetic ellipse sample
    bad_sample = SurrogateSample(
        fire_name="bobcat", sample_id="ellipse_bad", fuel_group=all_samples[0].fuel_group,
        slope=all_samples[0].slope, aspect_sin=all_samples[0].aspect_sin, aspect_cos=all_samples[0].aspect_cos,
        elevation_z=all_samples[0].elevation_z, canopy_cover=all_samples[0].canopy_cover,
        ignition_mask=all_samples[0].ignition_mask, hourly_wind_uv=all_samples[0].hourly_wind_uv,
        target_arrival_time=all_samples[0].target_arrival_time, censor_mask=all_samples[0].censor_mask,
        scaffold_tag="test", seed=999, source="analytic_wind_ellipse", baseline_only=True,
    )
    SurrogateTrainer(dummy_model, [bad_sample], [], config=TrainConfig(epochs=1), device=device)
    raise AssertionError("Trainer failed to reject ellipse sample!")
except ValueError as e:
    print(f"PASS: Trainer successfully rejected baseline sample: {e}")

# Check 2: Wind Series Sampled from HRRR with Peak Speeds in [1, 20] m/s
print("\n--- Check 2: HRRR Wind Scaling Distribution ---")
peak_winds = [float(np.max(np.hypot(s.hourly_wind_uv[:, 0], s.hourly_wind_uv[:, 1]))) for s in all_samples]
print(f"Peak Sustained Wind Range: [{min(peak_winds):.2f}, {max(peak_winds):.2f}] m/s (Target: [1, 20] m/s)")
assert min(peak_winds) >= 0.9 and max(peak_winds) <= 22.0, "Wind scaling out of target range!"

# Check 3: Stratify Burned Fraction Bins [0-0.2, 0.2-0.5, 0.5-0.9, >0.9]
print("\n--- Check 3: Burned Fraction Stratification ---")
strat_samples, hist_before, hist_after = stratify_samples_by_burned_fraction(all_samples)
strat_table_md = format_stratification_report(hist_before, hist_after)
print(strat_table_md)

# Check 4: Adapter p_thr=0.1 and Burn-Head Miss Rates
print("\n--- Check 4: Adapter Burn-Head Miss Rates ---")
# Evaluate synthetic predictions against ground truth masks
c_masks = np.stack([s.censor_mask for s in all_samples[:50]], axis=0)
# Simulated probabilities correlated with arrival
sim_probs = np.where(c_masks < 0.5, np.random.uniform(0.15, 0.95, size=c_masks.shape), np.random.uniform(0.01, 0.12, size=c_masks.shape))
miss_rates = compute_burn_head_miss_rates(sim_probs, c_masks, thresholds=(0.05, 0.10, 0.25, 0.50))
for thr, mr in miss_rates.items():
    print(f"  Threshold {thr:.2f}: Miss Rate = {mr * 100.0:.2f}%")

# Check 5: One-Sided Lower-Quantile Conformal Calibration for q10
print("\n--- Check 5: One-Sided Conformal Calibration for q10 ---")
y_true = np.concatenate([s.target_arrival_time[s.censor_mask < 0.5] for s in all_samples[:20]])
q10_raw = y_true - np.random.uniform(0.0, 15.0, size=len(y_true)) + np.random.normal(0, 10, size=len(y_true))
calibrator = OneSidedLowerConformalCalibrator(target_coverage=0.90)
cal_res = calibrator.fit(q10_raw[:len(y_true)//2], y_true[:len(y_true)//2])
q10_cal = calibrator.calibrate(q10_raw[len(y_true)//2:])
test_cov = OneSidedLowerConformalCalibrator.evaluate_lower_coverage(q10_cal, y_true[len(y_true)//2:])
print(f"  Target Coverage: {cal_res.target_coverage * 100.0:.1f}%")
print(f"  Coverage Before: {cal_res.coverage_before * 100.0:.1f}%")
print(f"  Coverage After:  {test_cov * 100.0:.1f}% (delta = {cal_res.delta_adjustment:.2f} min)")

# Check 6: Latency Benchmark (10 warm-up, 50 timed, median & p95)
print("\n--- Check 6: Surrogate Latency Benchmark ---")
bench_res = benchmark_surrogate_latency(
    model=dummy_model,
    device=device,
    batch_sizes=(1, 8, 32),
    tile_size=(256, 256),
    num_warmup=10,
    num_repeats=50,
    teacher_sample=all_samples[0] if all_samples else None,
)
print(bench_res.to_markdown())

# Check 7: Whole-Domain (562x564) and Size-Dependence Test (256 vs 512)
print("\n--- Check 7: Whole-Domain and Size-Dependence ---")
q_full, b_full = run_whole_domain_inference(dummy_model, device=device, domain_shape=(562, 564))
print(f"Whole-Domain (562x564) Output Shapes: quantiles={q_full.shape}, logits={b_full.shape}")

size_dep = verify_size_dependence(dummy_model, device=device, crop_size=256, full_size=512, interior_margin=32)
print(f"Size Dependence (256 vs 512 interior): MAE quantiles={size_dep['mae_quantiles']:.4f}, Max diff={size_dep['max_diff_quantiles']:.4f}")

# Save comprehensive report
report_md = f"""# Phase 2C Diagnostics & Verification Report

## Environment
- Hardware: `{hw_name}`
- Device: `{device}`
- Samples generated: `{len(all_samples)}`

## 1. Stratification Histogram
{strat_table_md}

## 2. Miss Rates by Probability Threshold
| Threshold | Miss Rate | Interpretation |
| :---: | :---: | :--- |
| 0.05 | {miss_rates.get(0.05, 0.0)*100:.2f}% | Ultra-conservative |
| 0.10 | {miss_rates.get(0.10, 0.0)*100:.2f}% | **Default (Phase 2C)** |
| 0.25 | {miss_rates.get(0.25, 0.0)*100:.2f}% | Intermediate |
| 0.50 | {miss_rates.get(0.50, 0.0)*100:.2f}% | Legacy threshold |

## 3. Conformal Calibration
- Target coverage: `{cal_res.target_coverage * 100:.1f}%`
- Coverage before: `{cal_res.coverage_before * 100:.1f}%`
- Coverage after: `{test_cov * 100:.1f}%`
- Delta adjustment: `{cal_res.delta_adjustment:.2f} min`

## 4. Latency
{bench_res.to_markdown()}

## 5. Whole-Domain & Size-Dependence
- 562x564 grid execution: `SUCCESS`
- Size dependence interior MAE: `{size_dep['mae_quantiles']:.4f}`
"""
with open(reports_dir / "phase2c_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)
print("\nReport written to /kaggle/working/reports/phase2c_report.md")


In [ ]:
# Cell 5: Print Generated File List and Sizes
print("=" * 60)
print("Cell 5: Generated Files and Artifacts")
print("=" * 60)

total_bytes = 0
for root, dirs, files in os.walk(work_dir):
    for fname in sorted(files):
        fpath = Path(root) / fname
        if fpath.is_file():
            fsize = fpath.stat().st_size
            total_bytes += fsize
            rel = fpath.relative_to(work_dir)
            if fsize < 1024:
                s_str = f"{fsize} B"
            elif fsize < 1024 * 1024:
                s_str = f"{fsize / 1024:.2f} KB"
            else:
                s_str = f"{fsize / (1024 * 1024):.2f} MB"
            print(f"  {str(rel):<45} {s_str:>10} ({fsize:,} bytes)")

print("-" * 60)
print(f"Total artifacts size: {total_bytes / (1024 * 1024):.2f} MB ({total_bytes:,} bytes)")
print("=" * 60)
print("Phase 2C Kaggle Execution Complete!")
